# Intelligent Employee Policy Assistant using RAG

## Problem Statement

A multinational company maintains thousands of employee-related documents, including HR policies, leave regulations, travel reimbursement guidelines, medical insurance policies, and work-from-home procedures.

Employees frequently ask questions about company policies. Searching through multiple PDF documents manually is time-consuming and inefficient.

Your task is to build a Retrieval-Augmented Generation (RAG) system that can answer employee questions accurately by retrieving relevant information from company policy documents and generating human-readable responses using a Large Language Model (LLM).

---

# Dataset

Create and use the following PDF documents:

- Employee Handbook.pdf
- Leave Policy.pdf
- Travel Policy.pdf
- Work From Home Policy.pdf
- Medical Insurance Policy.pdf

You may create sample content or use real HR policy documents.

---

# Task 1: Document Loading

## Objective

Load all company policy PDF documents.

## Requirements

- Read multiple PDF files.
- Extract text content from each PDF.
- Display document statistics.

## Deliverables

Display:

- File Name
- Number of Pages
- Total Characters
- Total Words

---

In [1]:
import pandas as pd

documents = {
    "Employee Handbook.pdf": """
    Welcome to the company.

    Employees must follow company policies.

    Professional conduct is expected.

    Equal opportunity employment is maintained.

    Training programs are available for all employees.
    """,

    "Leave Policy.pdf": """
    Employees receive 12 casual leaves annually.

    Employees receive 15 sick leaves annually.

    Unused casual leaves cannot be carried forward.
    """,

    "Travel Policy.pdf": """
    Travel expenses are reimbursed within 30 days.

    Original receipts must be submitted.

    International travel requires manager approval.
    """,

    "Work From Home Policy.pdf": """
    Employees may work from home twice per week.

    Manager approval is required for additional remote work.

    Employees must remain available during working hours.
    """,

    "Medical Insurance Policy.pdf": """
    All employees are covered under company medical insurance.

    Dependent coverage is available for spouse and children.

    Claims should be submitted within the prescribed timeline.
    """
}

stats = []

for file_name, text in documents.items():

    stats.append({
        "File Name": file_name,
        "Number of Pages": 1,
        "Total Characters": len(text),
        "Total Words": len(text.split())
    })

pd.DataFrame(stats)

,File Name,Number of Pages,Total Characters,Total Words
0,Employee Handbook.pdf,1,222,25
1,Leave Policy.pdf,1,155,19
2,Travel Policy.pdf,1,151,17
3,Work From Home Policy.pdf,1,175,23
4,Medical Insurance Policy.pdf,1,194,24


# Task 2: Document Chunking

## Objective

Split large policy documents into smaller chunks suitable for retrieval.

## Requirements

Implement:

### Fixed Size Chunking

### Recursive Chunking

Use:

- Chunk Size = 500
- Chunk Overlap = 100

## Deliverables

Display:

- Chunk Number
- Chunk Content
- Chunk Length

## Analysis

Explain:

- Why chunking is necessary in RAG.
- Advantages of chunking for retrieval.
- Differences between Fixed Size and Recursive Chunking.

---

In [2]:
!pip install langchain-text-splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter

chunk_size = 500
chunk_overlap = 100

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap
)

def fixed_chunk(text, size=500):

    chunks = []

    for i in range(0, len(text), size):
        chunks.append(text[i:i+size])

    return chunks


for file_name, text in documents.items():

    print("\n")
    print("="*80)
    print(file_name)
    print("="*80)

    print("\nFIXED SIZE CHUNKING")

    fixed_chunks = fixed_chunk(text)

    for i, chunk in enumerate(fixed_chunks):

        print(f"\nChunk Number: {i+1}")
        print("Chunk Length:", len(chunk))
        print(chunk)

    print("\nRECURSIVE CHUNKING")

    recursive_chunks = recursive_splitter.split_text(text)

    for i, chunk in enumerate(recursive_chunks):

        print(f"\nChunk Number: {i+1}")
        print("Chunk Length:", len(chunk))
        print(chunk)



Employee Handbook.pdf

FIXED SIZE CHUNKING

Chunk Number: 1
Chunk Length: 222

    Welcome to the company.

    Employees must follow company policies.

    Professional conduct is expected.

    Equal opportunity employment is maintained.

    Training programs are available for all employees.
    

RECURSIVE CHUNKING

Chunk Number: 1
Chunk Length: 212
Welcome to the company.

    Employees must follow company policies.

    Professional conduct is expected.

    Equal opportunity employment is maintained.

    Training programs are available for all employees.


Leave Policy.pdf

FIXED SIZE CHUNKING

Chunk Number: 1
Chunk Length: 155

    Employees receive 12 casual leaves annually.

    Employees receive 15 sick leaves annually.

    Unused casual leaves cannot be carried forward.
    

RECURSIVE CHUNKING

Chunk Number: 1
Chunk Length: 145
Employees receive 12 casual leaves annually.

    Employees receive 15 sick leaves annually.

    Unused casual leaves cannot be carried forwar

# Task 3: Embedding Generation

## Objective

Convert document chunks into vector representations using a Sentence Transformer model.

## Requirements

Use the embedding model:

- all-MiniLM-L6-v2

## Deliverables

Display:

- Chunk Content
- Embedding Shape
- Sample Embedding Values

## Analysis

Explain:

- How embeddings capture semantic meaning.
- Why embeddings are useful in RAG systems.
- How similar text produces similar embeddings.

In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

all_chunks = []

for file_name, text in documents.items():
    all_chunks.append(text)

chunk_embeddings = model.encode(all_chunks)

for chunk, embedding in zip(all_chunks, chunk_embeddings):

    print("\n" + "="*80)

    print("Chunk:")
    print(chunk[:200])

    print("\nEmbedding Shape:")
    print(embedding.shape)

    print("\nSample Embedding Values:")
    print(embedding[:10])

    print("="*80)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Chunk:

    Welcome to the company.

    Employees must follow company policies.

    Professional conduct is expected.

    Equal opportunity employment is maintained.

    Training programs are available f

Embedding Shape:
(384,)

Sample Embedding Values:
[-0.04832799  0.03488107  0.03899395  0.01822738 -0.00782879  0.04333074
  0.00286375 -0.11999402 -0.07735051 -0.04141947]

Chunk:

    Employees receive 12 casual leaves annually.

    Employees receive 15 sick leaves annually.

    Unused casual leaves cannot be carried forward.
    

Embedding Shape:
(384,)

Sample Embedding Values:
[ 0.06049041  0.03529654  0.02646347  0.02820184  0.06926834  0.05108616
  0.02678293  0.01582562 -0.08300046  0.02435355]

Chunk:

    Travel expenses are reimbursed within 30 days.

    Original receipts must be submitted.

    International travel requires manager approval.
    

Embedding Shape:
(384,)

Sample Embedding Values:
[ 4.7783531e-02  1.3307882e-02  4.0625747e-02  5.0407052e-02
  6.282

# Task 4: Build FAISS Vector Database

## Objective

Store document embeddings efficiently using a vector database.

## Requirements

- Create a FAISS index.
- Insert chunk embeddings into the index.

## Deliverables

Display:

- Number of Chunks
- Number of Stored Vectors
- Embedding Dimension

## Analysis

Explain:

- Why vector databases are required in RAG systems.
- How FAISS enables efficient similarity search.
- Benefits of storing embeddings instead of raw text.

In [4]:
!pip install faiss-cpu -q

import faiss
import numpy as np

embedding_dimension = chunk_embeddings.shape[1]

index = faiss.IndexFlatL2(embedding_dimension)

embeddings_np = np.array(chunk_embeddings).astype("float32")

index.add(embeddings_np)

print("Number of Chunks:", len(all_chunks))
print("Number of Stored Vectors:", index.ntotal)
print("Embedding Dimension:", embedding_dimension)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 94.3 MB/s eta 0:00:00
Number of Chunks: 5
Number of Stored Vectors: 5
Embedding Dimension: 384


# Task 5: Semantic Retrieval

## Objective

Retrieve the most relevant document chunks for employee questions using semantic similarity search.

## Example Queries

- How many casual leaves are available?
- Can employees work from home?
- How does travel reimbursement work?
- Who is covered under medical insurance?

## Deliverables

Display:

- Query
- Top 3 Retrieved Chunks
- Similarity Scores

## Analysis

Explain:

- Why those chunks were retrieved.
- How semantic similarity helps find relevant information.

In [5]:
queries = [
    "How many casual leaves are available?",
    "Can employees work from home?",
    "How does travel reimbursement work?",
    "Who is covered under medical insurance?"
]

query_embeddings = model.encode(queries)

for query, query_embedding in zip(queries, query_embeddings):

    print("\n")
    print("="*80)
    print("Query:", query)
    print("="*80)

    D, I = index.search(
        np.array([query_embedding]).astype("float32"),
        k=3
    )

    for rank, (idx, score) in enumerate(zip(I[0], D[0]), start=1):

        similarity = 1 / (1 + score)

        print(f"\nTop {rank}")

        print("Chunk:")
        print(all_chunks[idx][:300])

        print("Similarity Score:")
        print(round(similarity, 4))



Query: How many casual leaves are available?

Top 1
Chunk:

    Employees receive 12 casual leaves annually.

    Employees receive 15 sick leaves annually.

    Unused casual leaves cannot be carried forward.
    
Similarity Score:
0.5945

Top 2
Chunk:

    Employees may work from home twice per week.

    Manager approval is required for additional remote work.

    Employees must remain available during working hours.
    
Similarity Score:
0.3617

Top 3
Chunk:

    Welcome to the company.

    Employees must follow company policies.

    Professional conduct is expected.

    Equal opportunity employment is maintained.

    Training programs are available for all employees.
    
Similarity Score:
0.3353


Query: Can employees work from home?

Top 1
Chunk:

    Employees may work from home twice per week.

    Manager approval is required for additional remote work.

    Employees must remain available during working hours.
    
Similarity Score:
0.5991

Top 2
Chunk:

    All empl

# Task 6: Integrate Large Language Model

## Objective

Generate natural-language answers using retrieved document chunks.

## Model Used

TinyLlama

## Prompt Format

Context:
{retrieved_chunks}

Question:
{user_question}

Answer:

## Deliverables

Display:

- Retrieved Context
- Generated Answer

In [6]:
!pip install transformers accelerate torch -q

In [7]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    max_new_tokens=100
)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [8]:
for query, query_embedding in zip(queries, query_embeddings):

    D, I = index.search(
        np.array([query_embedding]).astype("float32"),
        k=3
    )

    retrieved_context = "\n".join(
        [all_chunks[idx] for idx in I[0]]
    )

    prompt = f"""
Context:
{retrieved_context}

Question:
{query}

Answer:
"""

    response = generator(
        prompt,
        do_sample=False
    )[0]["generated_text"]

    answer = response.split("Answer:")[-1].strip()

    print("\n")
    print("="*80)

    print("Question:")
    print(query)

    print("\nRetrieved Context:")
    print(retrieved_context[:500])

    print("\nGenerated Answer:")
    print(answer)

    print("="*80)

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=100) and `max_length



Question:
How many casual leaves are available?

Retrieved Context:

    Employees receive 12 casual leaves annually.

    Employees receive 15 sick leaves annually.

    Unused casual leaves cannot be carried forward.
    

    Employees may work from home twice per week.

    Manager approval is required for additional remote work.

    Employees must remain available during working hours.
    

    Welcome to the company.

    Employees must follow company policies.

    Professional conduct is expected.

    Equal opportunity employment is maintained.

    T

Generated Answer:
There are 12 casual leaves available annually.


[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)




Question:
Can employees work from home?

Retrieved Context:

    Employees may work from home twice per week.

    Manager approval is required for additional remote work.

    Employees must remain available during working hours.
    

    All employees are covered under company medical insurance.

    Dependent coverage is available for spouse and children.

    Claims should be submitted within the prescribed timeline.
    

    Welcome to the company.

    Employees must follow company policies.

    Professional conduct is expected.

    Equal oppor

Generated Answer:
Yes, employees may work from home twice per week.


[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)




Question:
How does travel reimbursement work?

Retrieved Context:

    Travel expenses are reimbursed within 30 days.

    Original receipts must be submitted.

    International travel requires manager approval.
    

    All employees are covered under company medical insurance.

    Dependent coverage is available for spouse and children.

    Claims should be submitted within the prescribed timeline.
    

    Employees receive 12 casual leaves annually.

    Employees receive 15 sick leaves annually.

    Unused casual leaves cannot be carried forward.
  

Generated Answer:
Travel expenses are reimbursed within 30 days. The original receipts must be submitted. International travel requires manager approval. All employees are covered under company medical insurance. Dependent coverage is available for spouse and children. Claims should be submitted within the prescribed timeline. Employees receive 12 casual leaves annually. Employees receive 15 sick leaves annually. Unused casual

# Task 7: Complete RAG Pipeline

## Objective

Build a reusable Employee Policy Assistant using Retrieval-Augmented Generation (RAG).

## Input

Employee Question

## Internal Flow

Question
↓
Embedding
↓
FAISS Search
↓
Top-K Chunks
↓
LLM
↓
Answer

## Output

Final Human-Readable Answer

## Deliverables

Display:

- User Question
- Retrieved Context
- Generated Answer

## Analysis

Explain:

- How the complete RAG pipeline works.
- Benefits of combining retrieval and generation.

In [9]:
def employee_policy_assistant(question):

    # Step 1: Generate Question Embedding
    question_embedding = model.encode([question])

    # Step 2: Retrieve Top-K Chunks
    D, I = index.search(
        np.array(question_embedding).astype("float32"),
        k=3
    )

    retrieved_chunks = []

    for idx in I[0]:
        retrieved_chunks.append(all_chunks[idx])

    context = "\n".join(retrieved_chunks)

    # Step 3: Build Prompt
    prompt = f"""
Context:
{context}

Question:
{question}

Answer:
"""

    # Step 4: Generate Answer
    response = generator(
        prompt,
        max_new_tokens=100,
        do_sample=False
    )[0]["generated_text"]

    answer = response.split("Answer:")[-1].strip()

    # Step 5: Display Results

    print("="*100)
    print("QUESTION")
    print("="*100)
    print(question)

    print("\n▼ RETRIEVED CONTEXT")
    print("-"*100)
    print(context)

    print("\n▼ GENERATED ANSWER")
    print("-"*100)
    print(answer)

    print("\n" + "="*100)

In [10]:
employee_policy_assistant(
    "Who is covered under medical insurance?"
)

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION
Who is covered under medical insurance?

▼ RETRIEVED CONTEXT
----------------------------------------------------------------------------------------------------

    All employees are covered under company medical insurance.

    Dependent coverage is available for spouse and children.

    Claims should be submitted within the prescribed timeline.
    

    Welcome to the company.

    Employees must follow company policies.

    Professional conduct is expected.

    Equal opportunity employment is maintained.

    Training programs are available for all employees.
    

    Employees receive 12 casual leaves annually.

    Employees receive 15 sick leaves annually.

    Unused casual leaves cannot be carried forward.
    

▼ GENERATED ANSWER
----------------------------------------------------------------------------------------------------
All employees are covered under company medical insurance.



In [11]:
employee_policy_assistant(
    "How many casual leaves are available?"
)

employee_policy_assistant(
    "Can employees work from home?"
)

employee_policy_assistant(
    "How does travel reimbursement work?"
)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION
How many casual leaves are available?

▼ RETRIEVED CONTEXT
----------------------------------------------------------------------------------------------------

    Employees receive 12 casual leaves annually.

    Employees receive 15 sick leaves annually.

    Unused casual leaves cannot be carried forward.
    

    Employees may work from home twice per week.

    Manager approval is required for additional remote work.

    Employees must remain available during working hours.
    

    Welcome to the company.

    Employees must follow company policies.

    Professional conduct is expected.

    Equal opportunity employment is maintained.

    Training programs are available for all employees.
    

▼ GENERATED ANSWER
----------------------------------------------------------------------------------------------------
There are 12 casual leaves available annually.



[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION
Can employees work from home?

▼ RETRIEVED CONTEXT
----------------------------------------------------------------------------------------------------

    Employees may work from home twice per week.

    Manager approval is required for additional remote work.

    Employees must remain available during working hours.
    

    All employees are covered under company medical insurance.

    Dependent coverage is available for spouse and children.

    Claims should be submitted within the prescribed timeline.
    

    Welcome to the company.

    Employees must follow company policies.

    Professional conduct is expected.

    Equal opportunity employment is maintained.

    Training programs are available for all employees.
    

▼ GENERATED ANSWER
----------------------------------------------------------------------------------------------------
Yes, employees may work from home twice per week.

QUESTION
How does travel reimbursement work?

▼ RETRIEVED CONTEXT
-------